In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import AdaBoostClassifier

df = pd.read_csv('../data/processed/bank_cleaned.csv')

X = df.drop(columns=['y', 'pdays'])
y = (df['y'] == 'yes').astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include='object').columns.tolist()

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

C:\Users\flowstxte\AppData\Local\Temp\ipykernel_10280\3449264131.py:19: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include='object').columns.tolist()


In [6]:
logreg_pipe = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

logreg_params = {'clf__C': [0.01, 0.1, 1, 10]}

logreg_grid = GridSearchCV(logreg_pipe, logreg_params, scoring='f1',
                            cv=StratifiedKFold(5), n_jobs=-1)
logreg_grid.fit(X_train, y_train)

print("Best params:", logreg_grid.best_params_)
print("Best CV F1:", logreg_grid.best_score_)

Best params: {'clf__C': 10}
Best CV F1: 0.4511221636256851


In [7]:
ada_pipe = Pipeline([
    ('prep', preprocessor),
    ('clf', AdaBoostClassifier(random_state=42))
])

ada_params = {
    'clf__n_estimators': [50, 100, 200],
    'clf__learning_rate': [0.5, 1.0]
}

ada_grid = GridSearchCV(ada_pipe, ada_params, scoring='f1',
                         cv=StratifiedKFold(5), n_jobs=-1)
ada_grid.fit(X_train, y_train)

print("Best params:", ada_grid.best_params_)
print("Best CV F1:", ada_grid.best_score_)

Best params: {'clf__learning_rate': 1.0, 'clf__n_estimators': 100}
Best CV F1: 0.3158040275728931


In [8]:
import joblib
joblib.dump(logreg_grid.best_estimator_, '../models/logreg_model.pkl')
joblib.dump(ada_grid.best_estimator_, '../models/adaboost_model.pkl')
print("Models saved.")

Models saved.
